# 04 - SQL Analysis

This notebook answers key business questions with SQL over the cleaned incident dataset.

**Business Goal:** Use SQL to extract operational insights (SLA, resolution times, workload distribution, and bottlenecks).

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Locate processed dataset whether notebook runs from project root or notebooks/
DATA_PATH = Path('../data/processed/incident_clean.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/processed/incident_clean.csv')

df = pd.read_csv(
    DATA_PATH,
    parse_dates=['opened_at', 'resolved_at', 'closed_at'],
    dtype={'vendor': 'string'},
    low_memory=False,
 )

con = sqlite3.connect(':memory:')
df.to_sql('incidents', con, if_exists='replace', index=False)

print({'dataset_path': str(DATA_PATH), 'shape': df.shape})

{'dataset_path': '..\\data\\processed\\incident_clean.csv', 'shape': (24918, 58)}


In [2]:
def run_sql(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, con)

run_sql("SELECT COUNT(*) AS total_incidents FROM incidents;")

,total_incidents
0,24918


## 1) Global KPI Snapshot

**Business Question:** What is the overall service performance level?

In [3]:
run_sql("""
SELECT
    COUNT(*) AS total_incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(closure_time_hours), 2) AS avg_closure_hours,
    ROUND(AVG(reassignment_count), 2) AS avg_reassignments,
    ROUND(AVG(reopen_count), 2) AS avg_reopens
FROM incidents;
""")

,total_incidents,sla_rate_pct,avg_resolution_hours,avg_closure_hours,avg_reassignments,avg_reopens
0,24918,63.44,178.17,315.33,0.94,0.01


## 2) Priority And SLA

**Business Question:** Which priority levels generate the most SLA risk?

In [4]:
run_sql("""
SELECT
    priority_text,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(reopen_count), 2) AS avg_reopens
FROM incidents
GROUP BY priority_text
ORDER BY incidents DESC;
""")

,priority_text,incidents,sla_rate_pct,avg_resolution_hours,avg_reopens
0,Moderate,23466,64.55,174.35,0.01
1,Low,774,84.11,283.19,0.00
2,High,408,0.74,152.46,0.01
3,Critical,270,2.22,266.08,0.00


## 3) Assignment Group Performance

**Business Question:** Which groups process most incidents and where are bottlenecks?

In [5]:
run_sql("""
SELECT
    assignment_group,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(reassignment_count), 2) AS avg_reassignments
FROM incidents
GROUP BY assignment_group
HAVING COUNT(*) >= 100
ORDER BY incidents DESC
LIMIT 15;
""")

,assignment_group,incidents,sla_rate_pct,avg_resolution_hours,avg_reassignments
0,Group 70,9444,83.84,50.22,0.50
1,NaN,2157,48.63,106.37,1.59
2,Group 25,1243,42.80,251.94,1.09
3,Group 39,1199,68.81,71.96,1.37
4,Group 24,1060,64.81,136.59,1.03
5,Group 23,811,58.69,245.22,1.58
6,Group 64,716,89.11,18.75,0.16
7,Group 73,576,53.99,124.71,1.02
8,Group 28,545,53.58,135.38,1.27
9,Group 27,518,58.30,123.84,0.99


## 4) Category Risk And Temporal Load

**Business Question:** Which categories and periods drive SLA pressure?

In [6]:
top_category_sla = run_sql("""
SELECT
    category,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct,
    ROUND(AVG(resolution_time_hours), 2) AS avg_resolution_hours
FROM incidents
GROUP BY category
HAVING COUNT(*) >= 100
ORDER BY sla_rate_pct ASC, incidents DESC
LIMIT 10;
""")

monthly_load = run_sql("""
SELECT
    strftime('%Y-%m', opened_at) AS opened_month,
    COUNT(*) AS incidents,
    ROUND(AVG(CAST(sla_compliant AS REAL)) * 100, 2) AS sla_rate_pct
FROM incidents
GROUP BY strftime('%Y-%m', opened_at)
ORDER BY opened_month;
""")

hourly_load = run_sql("""
SELECT
    CAST(strftime('%H', opened_at) AS INTEGER) AS opened_hour,
    COUNT(*) AS incidents
FROM incidents
GROUP BY CAST(strftime('%H', opened_at) AS INTEGER)
ORDER BY opened_hour;
""")

display(top_category_sla)
display(monthly_load)
display(hourly_load)

,category,incidents,sla_rate_pct,avg_resolution_hours
0,Category 55,106,20.75,674.02
1,Category 45,602,25.91,481.75
2,Category 34,501,33.53,1326.32
3,Category 40,436,38.99,152.81
4,Category 19,245,46.94,277.65
5,Category 61,810,47.04,182.37
6,Category 46,2432,48.52,316.02
7,Category 44,302,50.00,153.81
8,Category 23,1063,52.49,151.58
9,Category 24,640,52.66,279.68


,opened_month,incidents,sla_rate_pct
0,2016-02,207,64.73
1,2016-03,8995,39.88
2,2016-04,7934,76.70
3,2016-05,7508,77.56
4,2016-06,5,80.00
5,2016-07,14,78.57
6,2016-08,15,46.67
7,2016-09,12,58.33
8,2016-10,16,43.75
9,2016-11,26,65.38


,opened_hour,incidents
0,0,269
1,1,222
2,2,149
3,3,123
4,4,128
5,5,132
6,6,320
7,7,573
8,8,2547
9,9,3014
